<a href="https://colab.research.google.com/github/dudekulapeeravali2537sse-bot/ITA0616/blob/main/Exp_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import math
from collections import Counter

# Play Tennis Dataset
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain',
                'Rain', 'Rain', 'Overcast', 'Sunny',
                'Sunny', 'Rain', 'Sunny', 'Overcast',
                'Overcast', 'Rain'],

    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild',
                    'Cool', 'Cool', 'Cool', 'Mild',
                    'Cool', 'Mild', 'Mild', 'Mild',
                    'Hot', 'Mild'],

    'Humidity': ['High', 'High', 'High', 'High',
                 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'High'],

    'Wind': ['Weak', 'Strong', 'Weak', 'Weak',
             'Weak', 'Strong', 'Strong', 'Weak',
             'Weak', 'Weak', 'Strong', 'Strong',
             'Weak', 'Strong'],

    'PlayTennis': ['No', 'No', 'Yes', 'Yes',
                   'Yes', 'No', 'Yes', 'No',
                   'Yes', 'Yes', 'Yes', 'Yes',
                   'Yes', 'No']
}

df = pd.DataFrame(data)

# Calculate Entropy
def entropy(target):
    counts = Counter(target)
    total = len(target)

    ent = 0
    for count in counts.values():
        p = count / total
        ent -= p * math.log2(p)

    return ent

# Calculate Information Gain
def information_gain(df, attribute, target):
    total_entropy = entropy(df[target])
    total = len(df)

    weighted_entropy = 0

    for value in df[attribute].unique():
        subset = df[df[attribute] == value]
        weighted_entropy += (
            len(subset) / total
        ) * entropy(subset[target])

    return total_entropy - weighted_entropy

# ID3 Algorithm
def id3(df, attributes, target):

    # If all examples belong to one class
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    # If no attributes remain
    if not attributes:
        return df[target].mode()[0]

    # Select attribute with maximum information gain
    best_attribute = max(
        attributes,
        key=lambda attr: information_gain(df, attr, target)
    )

    tree = {best_attribute: {}}

    remaining_attributes = [
        attr for attr in attributes if attr != best_attribute
    ]

    # Create branches
    for value in df[best_attribute].unique():

        subset = df[df[best_attribute] == value]

        tree[best_attribute][value] = id3(
            subset,
            remaining_attributes,
            target
        )

    return tree

# Build Decision Tree
attributes = list(df.columns[:-1])

tree = id3(df, attributes, 'PlayTennis')

print("Decision Tree:")
print(tree)

# Classification of new sample
def predict(tree, sample):

    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample[attribute]

    if value not in tree[attribute]:
        return "Unknown"

    return predict(tree[attribute][value], sample)


# New sample
new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

result = predict(tree, new_sample)

print("\nNew Sample:", new_sample)
print("Classification:", result)


Decision Tree:
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}

New Sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Classification: No
